# SafeHire MCP demo



In [ ]:
from pathlib import Path

import gradio as gr
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio

load_dotenv(override=True)

INSTRUCTIONS = """
You are a SafeHire risk evaluation assistant.

When user provides:
- name
- months of experience
- complaints

You MUST call the calculate_risk tool.
Then explain the result in plain English.
"""
MODEL = "gpt-4.1-mini"


def _mcp_cwd() -> Path:
    for base in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (base / "safehire_server.py").exists():
            return base
    return Path.cwd().resolve()


def _stdio_params():
    return {
        "command": "uv",
        "args": ["run", "safehire_server.py"],
        "cwd": str(_mcp_cwd()),
    }


async def evaluate_safehire(name: str, months: float, complaints: float) -> str:
    name = (name or "").strip()
    if not name:
        return "Please enter a name."
    try:
        m = int(months)
        c = int(complaints)
    except (TypeError, ValueError):
        return "Months and complaints must be numbers."
    text = (
        f"What is the risk for {name} with {m} months of experience "
        f"and {c} complaint(s)?"
    )
    async with MCPServerStdio(
        params=_stdio_params(), client_session_timeout_seconds=60
    ) as mcp_server:
        agent = Agent(
            name="safehire_assistant",
            instructions=INSTRUCTIONS,
            model=MODEL,
            mcp_servers=[mcp_server],
        )
        with trace("safehire_assistant"):
            result = await Runner.run(agent, text, max_turns=20)
        return result.final_output


with gr.Blocks(title="SafeHire Risk Assistant") as safehire_demo:
    gr.Markdown(
        "## SafeHire risk assistant\n"
        "Fill in the fields below, then **Evaluate**."
    )
    with gr.Row():
        sh_name = gr.Textbox(label="Name", placeholder="John Doe", scale=2)
        sh_months = gr.Number(label="Months experience", value=10, minimum=0, step=1)
        sh_complaints = gr.Number(label="Complaints", value=1, minimum=0, step=1)
    run_safehire = gr.Button("Evaluate", variant="primary")
    safehire_out = gr.Markdown(label="Assistant reply")

    run_safehire.click(
        evaluate_safehire,
        inputs=[sh_name, sh_months, sh_complaints],
        outputs=[safehire_out],
    )

safehire_demo.launch(inline=True, inbrowser=False)